# 01_schema_results — Return complex objects validated by JSON Schema

Sometimes a result's shape is awkward as a deep tree of nested Pydantic classes — an audit trail, a metadata blob, a partial entity view. Two field-level tools let one field hold plain JSON validated against an explicit **strict JSON Schema**, while the rest of the model stays ordinarily typed. Both are plain Pydantic v2 types, so they validate the same way for a service return AND for in-code construction — no machine required.

**What's new**

| Concept | Description |
|---------|-------------|
| `JsonSchemaValue.define(name=, schema=)` | A field type holding arbitrary JSON validated by a strict schema |
| strict-by-construction | objects list `properties` + `additionalProperties: false`; arrays declare `items` (else `define` raises) |
| `model_dump()` / `model_json_schema()` | raw JSON out / the schema surfaced to FastAPI & MCP |
| `BaseEntity.schema(schema={...})` | A partial entity projection — a validated `dict`, not a hydrated entity |

In [ ]:
!pip install aoa-action-machine

In [ ]:
from pydantic import Field, ValidationError

from aoa.action_machine.domain import BaseEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.model import BaseResult, JsonSchemaValue

## 1. A complex object as one schema-validated field

The schema is **strict**: every object lists `properties` and sets `additionalProperties: false`; every array declares `items`. `define` rejects a loose schema immediately. Define the type at module level so its identity is stable.

In [ ]:
AUDIT_SCHEMA = {
    "type": "object",
    "properties": {
        "actor": {"type": "string"},
        "changes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "field": {"type": "string"},
                    "to": {"type": "string"},
                },
                "required": ["field", "to"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["actor", "changes"],
    "additionalProperties": False,
}
AuditReport = JsonSchemaValue.define(name="AuditReport", schema=AUDIT_SCHEMA)


class ChangeAuditResult(BaseResult):
    entity_id: str = Field(description="Affected entity")
    audit: AuditReport = Field(description="Audit trail payload (validated by JSON Schema)")

## 3. A partial projection of an entity

`BaseEntity.schema(schema=...)` binds a field to an entity class (for readers and the graph) but validates it on the wire against an explicit subset schema. The value is a plain validated `dict` — not a hydrated `OrderEntity` (no lifecycle, relations, or `FieldNotLoadedError`).

In [ ]:
class ShopDomain(BaseDomain):
    name = "shop"
    description = "Shop domain"


@entity(description="Customer order", domain=ShopDomain)
class OrderEntity(BaseEntity):
    id: str = Field(description="Order id")
    status: str = Field(description="Order status")
    total: float = Field(description="Order total")
    customer_email: str = Field(description="Full-entity field, omitted on the wire")


# The wire projection returns only id/status/total — not customer_email.
_ORDER_WIRE = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "status": {"type": "string"},
        "total": {"type": "number"},
    },
    "required": ["id", "status", "total"],
    "additionalProperties": False,
}


class OrderSummaryResult(BaseResult):
    order: OrderEntity.schema(schema=_ORDER_WIRE) = Field(  # type: ignore[valid-type]
        description="Partial order projection (no nested customer)",
    )

## Run

A complex object validated by schema (and its schema exposed to FastAPI/MCP), the same type validated in-code, and a partial entity projection that rejects fields outside the schema.

> This example is synchronous — no `await` needed.

In [ ]:
def main() -> None:
    result = ChangeAuditResult(
        entity_id="ord-1",
        audit={"actor": "alice", "changes": [{"field": "status", "to": "paid"}]},
    )
    print("1) Service return — complex object validated by JSON Schema:")
    print(f"   model_dump() -> {result.model_dump()}")
    audit_schema = ChangeAuditResult.model_json_schema()["properties"]["audit"]
    print(f"   schema exposed to FastAPI/MCP for `audit` -> type={audit_schema.get('type')}, "
          f"required={audit_schema.get('required')}")

    print("\n2) Same type used in-code (model_validate):")
    ok = ChangeAuditResult.model_validate(
        {"entity_id": "ord-2", "audit": {"actor": "bob", "changes": []}}
    )
    print(f"   valid payload   -> accepted ({ok.entity_id})")
    try:
        ChangeAuditResult.model_validate(
            {"entity_id": "ord-3", "audit": {"actor": "eve", "changes": [{"field": "status"}]}}
        )
    except ValidationError:
        print("   invalid payload -> ValidationError (a change item is missing required 'to')")

    print("\n3) Partial entity projection (BaseEntity.schema):")
    summary = OrderSummaryResult.model_validate(
        {"order": {"id": "o1", "status": "paid", "total": 99.5}}
    )
    print(f"   partial order   -> accepted: {summary.order}")
    try:
        OrderSummaryResult.model_validate(
            {"order": {"id": "o2", "status": "paid", "total": 10.0, "secret": "x"}}
        )
    except ValidationError:
        print("   unexpected field -> rejected (schema forbids fields outside the projection)")


main()